In [9]:
import os
import re
import joblib
import numpy as np
import pandas as pd
from textblob import TextBlob
from scipy.sparse import hstack
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Load the RAW dataset instead of the engineered one
raw_df = pd.read_csv("../Dataset/archive (3)/mental_heath_unbanlanced.csv")

# Drop any rows where text or status might be missing
raw_df = raw_df.dropna(subset=['text', 'status']).copy()

# ==========================================
# NEW CODE: INJECT POSITIVE EXAMPLES
# ==========================================
positive_texts = [
    "I am happy", "I am feeling great today", "Life is wonderful",
    "I am so joyful", "Everything is going perfectly", "I feel amazing",
    "I am having a great day", "I am feeling good", "I love my life", 
    "I am doing well", "I am extremely happy", "Today is a beautiful day"
] * 100  # Multiply by 100 to generate 1,200 positive records

synthetic_data = pd.DataFrame({
    'text': positive_texts,
    'status': ['Normal'] * len(positive_texts)
})

# Append the synthetic data to the main dataframe
raw_df = pd.concat([raw_df, synthetic_data], ignore_index=True)
# ==========================================

# For demonstration and memory management, you might want to remove duplicates
raw_df = raw_df.drop_duplicates(subset=['text']).copy()
raw_df.reset_index(drop=True, inplace=True)

print(f"Total raw records to process: {len(raw_df)}")

Total raw records to process: 48957


In [2]:
def extract_features_from_text(text):
    text = str(text)
    words = text.split()
    
    text_length = len(text)
    word_count = len(words)
    
    avg_word_length = sum(len(word) for word in words) / word_count if word_count > 0 else 0
    num_urls = len(re.findall(r'https?://\S+|www\.\S+', text))
    num_emojis = len(re.findall(r'[\U0001F300-\U0001FAFF]', text))
    num_special_chars = len(re.findall(r'[^a-zA-Z0-9\s]', text))
    num_excess_punct = len(re.findall(r'[!?.,]{2,}', text))
    
    stopwords = {
        'a', 'an', 'the', 'and', 'or', 'but', 'if', 'is', 'are', 'was', 'were',
        'to', 'of', 'in', 'on', 'for', 'with', 'as', 'at', 'by', 'from',
        'it', 'this', 'that', 'i', 'you', 'he', 'she', 'we', 'they'
    }
    
    stopword_count = sum(1 for word in words if word.lower() in stopwords) if word_count > 0 else 0
    stopword_ratio = stopword_count / word_count if word_count > 0 else 0
    
    unique_words = set(word.lower() for word in words)
    type_token_ratio = len(unique_words) / word_count if word_count > 0 else 0
    
    # Sentiment
    sentiment = TextBlob(text).sentiment
    polarity = sentiment.polarity
    subjectivity = sentiment.subjectivity
    
    # POS tags
    tagged_words = TextBlob(text).tags
    if len(tagged_words) > 0:
        total_tags = len(tagged_words)
        noun_ratio = sum(1 for _, tag in tagged_words if tag.startswith("NN")) / total_tags
        verb_ratio = sum(1 for _, tag in tagged_words if tag.startswith("VB")) / total_tags
        adj_ratio = sum(1 for _, tag in tagged_words if tag.startswith("JJ")) / total_tags
        adv_ratio = sum(1 for _, tag in tagged_words if tag.startswith("RB")) / total_tags
    else:
        noun_ratio = verb_ratio = adj_ratio = adv_ratio = 0
    
    # Keywords
    text_lower = text.lower()
    suicidal_keywords = ["suicide", "suicidal", "kill myself", "kill me", "want to die", "wanna die", "end my life", "end it all", "take my life", "overdose", "self harm", "self-harm"]
    stress_keywords = ["stress", "stressed", "pressure", "overwhelmed", "burnout", "exhausted", "overworked"]
    help_keywords = ["help me", "need help", "someone help", "need support", "looking for help", "therapy", "therapist", "counselling", "counseling"]
    
    has_suicidal_keyword = int(any(k in text_lower for k in suicidal_keywords))
    has_stress_keyword = int(any(k in text_lower for k in stress_keywords))
    has_help_keyword = int(any(k in text_lower for k in help_keywords))
    
    return {
        "text_length": text_length, "word_count": word_count, "num_urls": num_urls, 
        "num_emojis": num_emojis, "num_special_chars": num_special_chars, "num_excess_punct": num_excess_punct, 
        "avg_word_length": avg_word_length, "stopword_ratio": stopword_ratio, "type_token_ratio": type_token_ratio, 
        "polarity": polarity, "subjectivity": subjectivity, "noun_ratio": noun_ratio, 
        "verb_ratio": verb_ratio, "adj_ratio": adj_ratio, "adv_ratio": adv_ratio, 
        "has_suicidal_keyword": has_suicidal_keyword, "has_stress_keyword": has_stress_keyword, "has_help_keyword": has_help_keyword
    }

In [4]:
print("Extracting features... This may take a while depending on dataset size.")

# Apply the function to all text rows
extracted_features_list = raw_df['text'].apply(extract_features_from_text)

# Convert the list of dictionaries into a DataFrame
features_df = pd.DataFrame(extracted_features_list.tolist())

# Check the new raw scales (text_length should now have normal character counts, not 0 to 1)
print(features_df[['text_length', 'word_count']].describe())

Extracting features... This may take a while depending on dataset size.
        text_length    word_count
count  48945.000000  48945.000000
mean     400.257309     78.455123
std      620.597810    122.758607
min        7.000000      1.000000
25%       74.000000     14.000000
50%      243.000000     47.000000
75%      559.000000    110.000000
max    38785.000000   9684.000000


In [5]:
# Target and Text
X_text = raw_df['text']
y = raw_df['status']

# Split data
X_text_train, X_text_test, X_feat_train, X_feat_test, y_train, y_test = train_test_split(
    X_text, features_df, y, test_size=0.20, random_state=42, stratify=y
)

# 1. Fit and Transform TF-IDF
tfidf = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1,2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)
X_train_tfidf = tfidf.fit_transform(X_text_train)
X_test_tfidf = tfidf.transform(X_text_test)

# 2. Fit and Transform StandardScaler on RAW generated features
scaler = StandardScaler()
X_train_feat_scaled = scaler.fit_transform(X_feat_train)
X_test_feat_scaled = scaler.transform(X_feat_test)

# 3. Combine TF-IDF and Scaled Features
X_train_hybrid = hstack([X_train_tfidf, X_train_feat_scaled]).tocsr()
X_test_hybrid = hstack([X_test_tfidf, X_test_feat_scaled]).tocsr()

print("Hybrid training shape:", X_train_hybrid.shape)

Hybrid training shape: (39156, 20018)


In [7]:
# Train Logistic Regression
print("Training hybrid model...")
hybrid_model = LogisticRegression(
    max_iter=1000, 
    class_weight="balanced", 
    random_state=42
)
hybrid_model.fit(X_train_hybrid, y_train)

# Evaluate
y_pred = hybrid_model.predict(X_test_hybrid)
print("\nModel Evaluation on Test Set:")
print(classification_report(y_test, y_pred))

Training hybrid model...

Model Evaluation on Test Set:
              precision    recall  f1-score   support

     Anxiety       0.75      0.87      0.80      1065
  Depression       0.75      0.67      0.71      2853
      Normal       0.92      0.93      0.92      3630
    Suicidal       0.71      0.74      0.72      2241

    accuracy                           0.80      9789
   macro avg       0.78      0.80      0.79      9789
weighted avg       0.80      0.80      0.80      9789



In [8]:
# Ensure output directory exists
models_path = "../Models"
os.makedirs(models_path, exist_ok=True)

# Save the updated pipeline components
joblib.dump(tfidf, os.path.join(models_path, "tfidf_vectorizer.pkl"))
joblib.dump(scaler, os.path.join(models_path, "scaler.pkl"))
joblib.dump(hybrid_model, os.path.join(models_path, "hybrid_model.pkl"))

print("Corrected models saved successfully! You can now run your GUI.")

Corrected models saved successfully! You can now run your GUI.
